In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
from random import randint

from rich.pretty import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import ignite_classes as ic

## Load Data

In [ ]:
binary_columns_tot = ["human", "anthropic", "vegetation", "rock", "snow", "water", "zoom", "animal", "no"]
binary_columns_todrop = ["animal", "no", "water", "zoom"]
binary_columns = ["human", "anthropic", "vegetation", "rock", "snow"]
csv_raw = pd.read_csv(
    Path(".").joinpath("data").joinpath("annotations.csv")
).drop(
    binary_columns_todrop, 
    axis=1
)
csv_raw

In [ ]:
csv_raw.sum()

In [ ]:
n = 2000

rng = np.random.default_rng(42)

# ------------------------------------------------------------------ #
# Compute weights from imbalance in the original dataframe
# ------------------------------------------------------------------ #
# p = proportion of 1s; distance from 0.5 ranges from 0 (perfect balance)
# to 0.5 (all 0s or all 1s). We map it to a weight >= 1.
# weight = 1 + k * (|p - 0.5| / 0.5)  with k controlling the max weight.
k = 4  # max additional weight on top of the baseline 1
weights = {}
#print("\nAuto-computed column weights:")
for col in binary_columns:
    p = csv_raw[col].mean()
    imbalance = abs(p - 0.5) / 0.5   # 0 = perfectly balanced, 1 = fully skewed
    weights[col] = 1 + k * imbalance
    #print(f"  {col}: proportion of 1s = {p:.3f}, weight = {weights[col]:.2f}")

# ------------------------------------------------------------------ #
# Greedy balanced selection
# ------------------------------------------------------------------ #
df_shuffled = csv_raw.sample(frac=1, random_state=int(rng.integers(1e6))).reset_index(drop=True)

selected_indices = []
counts = {col: {0: 0, 1: 0} for col in binary_columns}
target_per_class = n // 2

for _, row in df_shuffled.iterrows():
    if len(selected_indices) >= n:
        break

    score = 0
    for col in binary_columns:
        w = weights[col]
        val = int(row[col])
        current = counts[col][val]
        current_opposite = counts[col][1 - val]

        if current < target_per_class:
            score += 1 * w
        elif current >= target_per_class and current_opposite < target_per_class:
            score -= 1 * w

    if score >= 0:
        selected_indices.append(row.name)
        for col in binary_columns:
            counts[col][int(row[col])] += 1

csv = df_shuffled.loc[selected_indices].reset_index(drop=True)

pd.DataFrame(data={
    "var":[col for col in binary_columns],
    "per":[csv[col].mean()*100 for col in binary_columns],
    "qtt":[sum(csv[col]) for col in binary_columns],
    "all":len(csv),
}).sort_values("per", ascending = False)



## Check labels

In [ ]:
pprint(csv.columns[2:])

## Test Datasets

In [ ]:
path_to_images = Path(".").joinpath("data").joinpath("images")
path_to_images.is_dir()

In [ ]:
dataset = ic.FldDataset(data = csv, train_mode=True,test_mode=True)

In [ ]:
pprint(dataset.transform)
rnd_data = dataset[randint(0, len(dataset) - 1)]
pprint(rnd_data["labels"])
plt.imshow(rnd_data["image"])

In [ ]:
plt.imshow(dataset[100]["image"])

## Train

### Split dataset

In [ ]:
csv_strat = csv.copy()
csv_strat["strat"] = ""
for col in csv.columns[2:]:
    print(col)
    csv_strat["strat"] += csv_strat[col].astype(str)

pprint(csv_strat.strat.value_counts())

csv_strat

In [ ]:
csv_strat_balanced = csv_strat.copy()
csv_strat_count = pd.DataFrame(csv_strat.strat.value_counts()).reset_index()
strat_tokeep = csv_strat_count[csv_strat_count['count'] >= 10]
csv_strat_balanced = csv_strat_balanced[csv_strat_balanced['strat'].isin(strat_tokeep['strat'])]
pprint(csv_strat_balanced.strat.value_counts())

In [ ]:
trainval, test = train_test_split(csv_strat_balanced, test_size = 0.15, random_state = 42, stratify=csv_strat_balanced["strat"])
train, val = train_test_split(trainval, test_size = 0.18, stratify=trainval["strat"])

train = train.drop("strat", axis=1)
val = val.drop("strat", axis=1)
test = test.drop("strat", axis=1)

for n,d in [("train",train), ("val",val), ("test",test)]:
    print(n, d.shape)

In [ ]:
ic.FocalLoss()

In [ ]:
model = ic.train_model(
    train_data=train,
    val_data=val,
    batch_size=64,
    max_epochs=200,
    image_size=224,
    backbone="hf_swt_t",
    loss_name="bce",
    loss_params={"alpha": 0.5, "gamma": 1},
    device=ic.get_device(),
    checkpoints_n_saved=1,
    learning_rate=0.00001,
    early_stoper_patience=10,
    early_stoper_min_delta=0.001,
    use_lr_finder=False,
    lr_scheduler_step=10,
    lr_scheduler_gamma=0.85,
    print_steps="print",
    log_progress=True,
    plot_loss=False,
    num_workers=10,
)

In [ ]:
model.get_val_data(dataset=ic.FldDataset(data=val, train_mode=False))["classification_report"]